# SSE Clock Plus - HTMX v4

## What this app does

This app displays a live-updating clock and a counter, both powered by a single SSE (Server-Sent Events) stream.

## htmx v2 approach

The original htmx v2 version (`sse_clock_plus.py`) uses a **custom named event** (`TimeUpdateEvent`) for SSE swapping:

- The server sends `event: TimeUpdateEvent\ndata: ...\n\n`
- The client uses `sse_swap="TimeUpdateEvent"` to swap content when that specific event arrives
- This pattern enables **multiplexing**: a single SSE connection can dispatch different events to different elements using `sse_swap="EventA"`, `sse_swap="EventB"`, etc.

## htmx v4 approach

In htmx v4, SSE uses the `hx-sse` extension (`exts='sse'` in `fast_app`). The semantics of named events changed:

- **Named events** (`event: ...`) now trigger a **DOM event** instead of performing a swap. This is useful for JavaScript-driven logic, not declarative swaps.
- **Unnamed events** (just `data:` lines) perform normal htmx swaps into the target.

For dispatching content to multiple elements, htmx v4 introduces **`<hx-partial>`** tags. Each partial specifies its own `hx-target`, so a single `data:` line can update multiple parts of the page:

```html
<hx-partial hx-target="#clock">12:30:45</hx-partial>
<hx-partial hx-target="#counter">42</hx-partial>
```

## Changes from v2

1. **SSE extension via `exts='sse'`**, uses htmx v4's built-in `hx-sse` extension instead of the separate v2 extension script
2. **Standard htmx attributes**, use `hx_get="/time-sender"` + `hx_trigger="load"` to initiate the stream
3. **`<hx-partial>` for multi-target updates**, each element targeted by its own partial, replacing the named event multiplexing pattern
4. **`sse_message(..., htmx4=True)`**, omits the `event:` line so htmx performs a normal swap

In [1]:
from fasthtml.common import *
from fasthtml.jupyter import *


In [2]:
import asyncio
from datetime import datetime

app, rt = fast_app(htmx=False, htmx4=True, exts='sse')

@rt("/")
def get():
    return Titled("SSE Clock Plus",
            Div(hx_get="/time-sender", hx_trigger="load")(
                Div("Time: ", Span("XX:XX", id="time")),
                Div("Count: ", Span("0", id="counter"))
            )
        )

async def time_generator():
    counter = 0
    while True:
        _time = Span(datetime.now().strftime('%H:%M:%S'))
        counter += 1
        yield sse_message(
            (HxPartial(_time, hx_target="#time"), HxPartial(counter, hx_target="#counter")),
            htmx4=True)
        await asyncio.sleep(1)

@rt("/time-sender")
async def get():
    return EventStream(time_generator())

srv = JupyUvi(app)